In [ ]:
from pathlib import Path
import json

def resolve_assignment_dir():
    cwd = Path.cwd().resolve()

    # Local/GPU-lab checkout.
    for p in [cwd, *cwd.parents]:
        if p.name == "GPU_Assignment1" or (p / "README_FIRST.md").exists():
            return p

    # Google Colab / Google Drive.
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        root = Path("/content/drive/MyDrive")
        matches = list(root.rglob("GPU_Assignment1"))
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(
                "Multiple GPU_Assignment1 folders found: "
                + ", ".join(str(x) for x in matches)
            )
    except ImportError:
        pass

    raise FileNotFoundError(
        "Could not locate GPU_Assignment1. Run from the assignment folder "
        "or mount Google Drive."
    )

ASSIGNMENT_DIR = resolve_assignment_dir()
ARTIFACT_DIR = ASSIGNMENT_DIR / "hw2_5_artifacts"
PARAM_FILE = ARTIFACT_DIR / "student_params.json"

if not PARAM_FILE.exists():
    raise FileNotFoundError(f"student_params.json not found at: {PARAM_FILE}")

PARAMS = json.loads(PARAM_FILE.read_text())
SID4 = PARAMS["SID4"]
SEED = PARAMS["SEED"]
SLICE = PARAMS["SLICE"]
HP_ID = PARAMS["HP_ID"]
CLS_A = PARAMS["CLS_A"]
CLS_B = PARAMS["CLS_B"]

print("Loaded:", PARAM_FILE)


# HW2.5 — Notebook 01
## Part B: Precision / achieved throughput
## Part C: Bandwidth-bound vs compute-bound

Run after Notebook 00, on the **same reserved GPU**. Every saved measurement contains the GPU UUID.

The benchmark uses CUDA events, warm-up iterations, multiple repetitions, and reports median/mean/std/CV. FP16/BF16 reduced-precision reductions are disabled when the installed PyTorch exposes those switches, matching NVIDIA's FP32-accumulate Tensor peak used as the denominator.

In [ ]:
import os, sys, gc, math, time, json, random, traceback, subprocess, platform, re
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

FIG_DIR = ARTIFACT_DIR / "figures"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
RUN_LOG = ARTIFACT_DIR / "RUN_LOG.txt"

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def log(message=""):
    line = str(message)
    print(line)
    with RUN_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{utc_now()}] {line}\n")

def log_exception(prefix, exc):
    tb = traceback.format_exc()
    log(f"{prefix}: {type(exc).__name__}: {exc}")
    with RUN_LOG.open("a", encoding="utf-8") as f:
        f.write(tb + "\n")

def run_cmd(cmd, check=False):
    p = subprocess.run(cmd, capture_output=True, text=True)
    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed: {cmd}\nSTDOUT:\n{p.stdout}\nSTDERR:\n{p.stderr}"
        )
    return p

GPU_INFO_FILE = ARTIFACT_DIR / "gpu_info.json"
if not GPU_INFO_FILE.exists():
    raise FileNotFoundError("gpu_info.json missing. Run notebook 00 first.")

GPU_INFO = json.loads(GPU_INFO_FILE.read_text())
GPU_UUID = GPU_INFO["uuid"]
GPU_NAME = GPU_INFO["name"]
GPU_INDEX = int(GPU_INFO["nvidia_smi_index"])
CARD_SPECS = GPU_INFO["card_specs"]

if not GPU_INFO.get("hw25_supported_gpu", False):
    raise RuntimeError("Notebook 00 did not record an approved RTX 4090/5090 GPU.")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")

current = run_cmd([
    "nvidia-smi",
    "--query-gpu=index,name,uuid",
    "--format=csv,noheader,nounits"
], check=True)
current_rows = [[x.strip() for x in line.split(",")] for line in current.stdout.strip().splitlines()]
matching = [r for r in current_rows if len(r) == 3 and r[2] == GPU_UUID]
if not matching:
    raise RuntimeError(
        "The current GPU UUID does not match notebook 00. "
        "Rerun notebook 00 on this reserved GPU."
    )

DEVICE = torch.device("cuda:0")
torch.cuda.set_device(DEVICE)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

log("=" * 88)
log(
    f"Notebook started | GPU={GPU_NAME} | UUID={GPU_UUID} | "
    f"Driver CUDA={GPU_INFO.get('driver_cuda_version')} | "
    f"PyTorch CUDA build={torch.version.cuda}"
)


In [ ]:
# Precision benchmark configuration.
MATRIX_SIZES = [1024, 4096, 8192, 16384]
WARMUPS = 10
REPS_BY_N = {1024: 200, 4096: 100, 8192: 30, 16384: 10}

if hasattr(torch.backends.cuda.matmul, "allow_fp16_reduced_precision_reduction"):
    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
if hasattr(torch.backends.cuda.matmul, "allow_bf16_reduced_precision_reduction"):
    torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

def cuda_times_ms(fn, warmups, reps):
    with torch.inference_mode():
        for _ in range(warmups):
            fn()
        torch.cuda.synchronize()

        times = []
        for _ in range(reps):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            times.append(float(start.elapsed_time(end)))
    return np.asarray(times, dtype=float)

def trimmed_mean(values, trim_fraction=0.10):
    x = np.sort(np.asarray(values, dtype=float))
    k = int(len(x) * trim_fraction)
    if k == 0 or 2 * k >= len(x):
        return float(np.mean(x))
    return float(np.mean(x[k:-k]))

def benchmark_matmul(n, label, dtype, tf32_enabled):
    gc.collect()
    torch.cuda.empty_cache()

    torch.backends.cuda.matmul.allow_tf32 = bool(tf32_enabled)
    try:
        torch.set_float32_matmul_precision("high" if tf32_enabled else "highest")
    except Exception:
        pass

    a = torch.randn((n, n), device=DEVICE, dtype=dtype)
    b = torch.randn((n, n), device=DEVICE, dtype=dtype)

    c = torch.mm(a, b)
    del c
    torch.cuda.synchronize()

    reps = REPS_BY_N[n]
    times = cuda_times_ms(lambda: torch.mm(a, b), WARMUPS, reps)

    med_ms = float(np.median(times))
    mean_ms = float(np.mean(times))
    tmean_ms = trimmed_mean(times, 0.10)
    std_ms = float(np.std(times, ddof=1)) if len(times) > 1 else 0.0
    cv_pct = 100.0 * std_ms / mean_ms if mean_ms else float("nan")
    q1_ms, q3_ms = [float(v) for v in np.percentile(times, [25, 75])]
    iqr_ms = q3_ms - q1_ms

    flop = 2.0 * n**3
    achieved = flop / (med_ms / 1000.0) / 1e12
    peak = float(CARD_SPECS["theoretical_tflops_dense"][label])
    pct_peak = 100.0 * achieved / peak

    result = {
        "uuid": GPU_UUID,
        "gpu": GPU_NAME,
        "precision": label,
        "dtype": str(dtype),
        "N": n,
        "warmups": WARMUPS,
        "repetitions": reps,
        "median_ms": med_ms,
        "trimmed_mean_ms": tmean_ms,
        "mean_ms": mean_ms,
        "std_ms": std_ms,
        "cv_pct": cv_pct,
        "q1_ms": q1_ms,
        "q3_ms": q3_ms,
        "iqr_ms": iqr_ms,
        "achieved_tflops": achieved,
        "theoretical_tflops": peak,
        "pct_theoretical_peak": pct_peak,
    }

    log(
        f"Part B | UUID={GPU_UUID} | {label} N={n} | reps={reps} | "
        f"median={med_ms:.4f} ms | TFLOPS={achieved:.3f} | "
        f"%peak={pct_peak:.2f} | CV={cv_pct:.2f}% | IQR={iqr_ms:.4f} ms"
    )

    del a, b
    gc.collect()
    torch.cuda.empty_cache()
    return result

modes = [
    ("FP32", torch.float32, False),
    ("TF32", torch.float32, True),
    ("FP16", torch.float16, False),
    ("BF16", torch.bfloat16, False),
]

missing_peaks = [label for label, _, _ in modes if label not in CARD_SPECS["theoretical_tflops_dense"]]
if missing_peaks:
    raise RuntimeError(f"Missing vendor theoretical peaks for: {missing_peaks}")

rows = []
for label, dtype, tf32 in modes:
    for n in MATRIX_SIZES:
        try:
            rows.append(benchmark_matmul(n, label, dtype, tf32))
        except Exception as exc:
            log_exception(f"Part B failure | {label} N={n}", exc)
            rows.append({
                "uuid": GPU_UUID,
                "gpu": GPU_NAME,
                "precision": label,
                "dtype": str(dtype),
                "N": n,
                "warmups": WARMUPS,
                "repetitions": REPS_BY_N[n],
                "error": f"{type(exc).__name__}: {exc}",
            })
            gc.collect()
            torch.cuda.empty_cache()

part_b = pd.DataFrame(rows)
part_b.to_csv(ARTIFACT_DIR / "part_b_precision_results.csv", index=False)
display(part_b)

required_precisions = {"FP32", "TF32", "FP16", "BF16"}
successful_precisions = set(part_b.loc[part_b["achieved_tflops"].notna(), "precision"])
missing = sorted(required_precisions - successful_precisions)
if missing:
    raise RuntimeError(
        f"Required precision measurements are missing: {missing}. "
        "Do not substitute another precision."
    )


### Part B.3 — throughput figure and operational plateau

The x-axis is logarithmic so all required matrix sizes remain visible.

A precision is called **operationally plateaued** only when the first candidate size and every larger tested size are both:
- stable (`CV ≤ 5%`), and
- within a 10% throughput band across that tail.

If no tail satisfies both conditions, the notebook reports **“not reached”** instead of forcing a plateau from the maximum observed point.


In [ ]:
ok_b = part_b.dropna(subset=["achieved_tflops"]).copy()

fig, ax = plt.subplots(figsize=(9, 5.5))
for precision, grp in ok_b.groupby("precision"):
    grp = grp.sort_values("N")
    ax.plot(grp["N"], grp["achieved_tflops"], marker="o", label=precision)

ax.set_xscale("log", base=2)
ax.set_xticks(MATRIX_SIZES)
ax.set_xticklabels([str(n) for n in MATRIX_SIZES])
ax.set_xlabel("Matrix size N (log2 scale)")
ax.set_ylabel("Achieved TFLOPS")
ax.set_title(f"HW2.5 Part B — Dense Matmul Throughput\n{GPU_NAME} | {GPU_UUID}")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

plot_path = FIG_DIR / "part_b_tflops_vs_matrix_size.png"
fig.savefig(plot_path, dpi=180)
plt.show()
log(f"Saved {plot_path}")

PLATEAU_CV_MAX_PCT = 5.0
PLATEAU_TAIL_SPREAD_MAX_PCT = 10.0

plateau_rows = []
for precision, grp in ok_b.groupby("precision"):
    grp = grp.sort_values("N").reset_index(drop=True)

    plateau_n = None
    tail_spread_pct = None
    tail_max_cv = None

    for idx in range(len(grp)):
        tail = grp.iloc[idx:]
        tmin = float(tail["achieved_tflops"].min())
        tmax = float(tail["achieved_tflops"].max())
        tmean = float(tail["achieved_tflops"].mean())
        spread = 100.0 * (tmax - tmin) / tmean if tmean else np.inf
        max_cv = float(tail["cv_pct"].max())

        if spread <= PLATEAU_TAIL_SPREAD_MAX_PCT and max_cv <= PLATEAU_CV_MAX_PCT:
            plateau_n = int(grp.loc[idx, "N"])
            tail_spread_pct = spread
            tail_max_cv = max_cv
            break

    plateau_rows.append({
        "precision": precision,
        "observed_max_tflops": float(grp["achieved_tflops"].max()),
        "plateau_status": "reached" if plateau_n is not None else "not reached",
        "plateau_N": plateau_n,
        "tail_spread_pct": tail_spread_pct,
        "tail_max_cv_pct": tail_max_cv,
        "criterion": "tail spread <=10% and all tail CV <=5%",
    })

plateau_df = pd.DataFrame(plateau_rows)
plateau_df.to_csv(ARTIFACT_DIR / "part_b_plateaus.csv", index=False)
display(plateau_df)

for _, r in plateau_df.iterrows():
    if r["plateau_status"] == "reached":
        log(
            f"Part B plateau | {r['precision']} | N={int(r['plateau_N'])} | "
            f"tail spread={r['tail_spread_pct']:.2f}% | max CV={r['tail_max_cv_pct']:.2f}%"
        )
    else:
        log(
            f"Part B plateau | {r['precision']} | not reached within tested sizes. "
            "No plateau is claimed."
        )


### Part B.4 — lower-precision probe

The assignment says to run a lower precision **if the software stack exposes it**, or document what was tried and the failure. The next cell probes FP8 (`float8_e4m3fn`) with the installed PyTorch.

It first tries public `torch.mm`. If that fails, it tries the version-dependent internal `_scaled_mm` API. If neither is usable, the error is recorded as the tooling-maturity finding rather than fabricated.

In [ ]:
def try_fp8_backend():
    fp8 = getattr(torch, "float8_e4m3fn", None)
    errors = []
    if fp8 is None:
        return None, None, ["torch.float8_e4m3fn is not present in this PyTorch build."]

    a = torch.randn((256, 256), device=DEVICE, dtype=torch.float16).to(fp8)
    b = torch.randn((256, 256), device=DEVICE, dtype=torch.float16).to(fp8)

    try:
        z = torch.mm(a, b)
        torch.cuda.synchronize()
        del z
        return "torch.mm(float8_e4m3fn)", lambda x, y: torch.mm(x, y), errors
    except Exception as e:
        errors.append(f"torch.mm FP8 failed: {type(e).__name__}: {e}")

    scaled_mm = getattr(torch, "_scaled_mm", None)
    if scaled_mm is not None:
        scale = torch.tensor(1.0, device=DEVICE, dtype=torch.float32)

        def call_scaled(x, y):
            # PyTorch has changed this private signature across versions.
            attempts = [
                lambda: scaled_mm(x, y, scale_a=scale, scale_b=scale, out_dtype=torch.float16),
                lambda: scaled_mm(x, y, scale, scale, out_dtype=torch.float16),
                lambda: scaled_mm(x, y, scale_a=scale, scale_b=scale),
            ]
            last = None
            for f in attempts:
                try:
                    out = f()
                    return out[0] if isinstance(out, tuple) else out
                except Exception as exc:
                    last = exc
            raise last

        try:
            z = call_scaled(a, b)
            torch.cuda.synchronize()
            del z
            return "torch._scaled_mm(float8_e4m3fn)", call_scaled, errors
        except Exception as e:
            errors.append(f"torch._scaled_mm FP8 failed: {type(e).__name__}: {e}")
    else:
        errors.append("torch._scaled_mm is not present in this PyTorch build.")

    del a, b
    gc.collect()
    torch.cuda.empty_cache()
    return None, None, errors

backend_name, fp8_mm, fp8_errors = try_fp8_backend()
probe_lines = [
    f"GPU: {GPU_NAME}",
    f"UUID: {GPU_UUID}",
    f"PyTorch: {torch.__version__}",
    f"CUDA build: {torch.version.cuda}",
    f"Selected FP8 backend: {backend_name}",
]

fp8_rows = []
if backend_name is not None:
    log(f"FP8 exposed through: {backend_name}")
    fp8_dtype = torch.float8_e4m3fn
    # Keep this optional benchmark at the same required N values if the stack supports it.
    for n in MATRIX_SIZES:
        try:
            gc.collect()
            torch.cuda.empty_cache()
            a = torch.randn((n, n), device=DEVICE, dtype=torch.float16).to(fp8_dtype)
            b = torch.randn((n, n), device=DEVICE, dtype=torch.float16).to(fp8_dtype)
            reps = REPS_BY_N[n]
            times = cuda_times_ms(lambda: fp8_mm(a, b), WARMUPS, reps)
            med_ms = float(np.median(times))
            achieved = 2.0 * n**3 / (med_ms / 1000.0) / 1e12
            peak = CARD_SPECS["theoretical_tflops_dense"].get("FP8", np.nan)
            pct = 100.0 * achieved / peak if np.isfinite(peak) else np.nan
            fp8_rows.append({
                "uuid": GPU_UUID, "gpu": GPU_NAME, "precision": "FP8_e4m3fn",
                "backend": backend_name, "N": n, "repetitions": reps,
                "median_ms": med_ms, "achieved_tflops": achieved,
                "theoretical_tflops": peak, "pct_theoretical_peak": pct
            })
            log(f"Part B FP8 | UUID={GPU_UUID} | N={n} | TFLOPS={achieved:.3f} | %peak={pct:.2f}")
            del a, b
        except Exception as exc:
            msg = f"FP8 benchmark N={n} failed: {type(exc).__name__}: {exc}"
            fp8_errors.append(msg)
            log(msg)
            gc.collect()
            torch.cuda.empty_cache()
else:
    log("No usable FP8 matmul path was exposed by this software stack.")
    for e in fp8_errors:
        log("  " + e)

(ARTIFACT_DIR / "lower_precision_probe.txt").write_text("\n".join(probe_lines + fp8_errors) + "\n")

if fp8_rows:
    fp8_df = pd.DataFrame(fp8_rows)
else:
    fp8_df = pd.DataFrame([{
        "uuid": GPU_UUID,
        "gpu": GPU_NAME,
        "precision": "FP8_e4m3fn",
        "backend": backend_name,
        "status": "unavailable",
        "error": " | ".join(fp8_errors) if fp8_errors else "No usable FP8 path exposed",
    }])

fp8_df.to_csv(ARTIFACT_DIR / "part_b_fp8_results.csv", index=False)
display(fp8_df)

# Part C — bandwidth-bound vs compute-bound

Memory-bound operation: large FP32 elementwise add `c = a + b`.

- Useful FLOPs per element: 1 add
- Minimum bytes moved per element: 2 reads + 1 write = 12 bytes
- Arithmetic intensity = `1 / 12` FLOP/byte

Compute-bound operation: large FP16 square matrix multiplication.

For `C = A @ B`, counting the minimum read/write traffic once:
- FLOPs = `2N^3`
- bytes = `(A + B + C) = 3N^2 * 2 bytes`
- arithmetic intensity = `N/3` FLOP/byte

The roofline ridge point is `peak compute FLOP/s / peak memory byte/s`.

In [ ]:
# Size the vector so A, B, and C are large enough to stress DRAM without consuming the whole GPU.
total_vram_bytes = int(float(GPU_INFO["vram_mib"]) * 1024**2)
max_numel_by_budget = int((0.20 * total_vram_bytes) // (3 * 4))  # 20% VRAM, three FP32 arrays
NUMEL = min(256_000_000, max_numel_by_budget)
NUMEL = max(NUMEL, 32_000_000)

gc.collect()
torch.cuda.empty_cache()
a = torch.randn(NUMEL, device=DEVICE, dtype=torch.float32)
b = torch.randn(NUMEL, device=DEVICE, dtype=torch.float32)
c = torch.empty_like(a)

mem_times = cuda_times_ms(lambda: torch.add(a, b, out=c), warmups=5, reps=20)
mem_med_ms = float(np.median(mem_times))
bytes_per_add = NUMEL * 3 * 4
effective_bandwidth = bytes_per_add / (mem_med_ms / 1000.0) / 1e9
specified_bandwidth = float(CARD_SPECS["memory_bandwidth_GBs"])
bandwidth_pct = 100.0 * effective_bandwidth / specified_bandwidth
mem_ai = 1.0 / 12.0

log(
    f"Part C memory-bound | UUID={GPU_UUID} | elements={NUMEL} | median={mem_med_ms:.4f} ms | "
    f"effective={effective_bandwidth:.2f} GB/s | {bandwidth_pct:.2f}% of specified"
)
del a, b, c
gc.collect()
torch.cuda.empty_cache()

# Compute-bound case.
COMPUTE_N = 8192
a = torch.randn((COMPUTE_N, COMPUTE_N), device=DEVICE, dtype=torch.float16)
b = torch.randn((COMPUTE_N, COMPUTE_N), device=DEVICE, dtype=torch.float16)
comp_times = cuda_times_ms(lambda: torch.mm(a, b), warmups=3, reps=10)
comp_med_ms = float(np.median(comp_times))
comp_flops = 2.0 * COMPUTE_N**3
comp_tflops = comp_flops / (comp_med_ms / 1000.0) / 1e12
comp_bytes_min = 3.0 * COMPUTE_N**2 * 2.0
comp_ai = comp_flops / comp_bytes_min

peak_compute = float(CARD_SPECS["theoretical_tflops_dense"]["FP16"])
ridge_flop_per_byte = (peak_compute * 1e12) / (specified_bandwidth * 1e9)

mem_side = "bandwidth-bound side" if mem_ai < ridge_flop_per_byte else "compute-bound side"
comp_side = "bandwidth-bound side" if comp_ai < ridge_flop_per_byte else "compute-bound side"

part_c = {
    "uuid": GPU_UUID,
    "gpu": GPU_NAME,
    "memory_operation": "FP32 elementwise add",
    "memory_numel": NUMEL,
    "memory_median_ms": mem_med_ms,
    "effective_bandwidth_GBs": effective_bandwidth,
    "specified_bandwidth_GBs": specified_bandwidth,
    "bandwidth_pct_specified": bandwidth_pct,
    "memory_arithmetic_intensity_flop_per_byte": mem_ai,
    "memory_roofline_side": mem_side,
    "compute_operation": f"FP16 matmul N={COMPUTE_N}",
    "compute_median_ms": comp_med_ms,
    "compute_tflops": comp_tflops,
    "compute_arithmetic_intensity_flop_per_byte": comp_ai,
    "compute_roofline_side": comp_side,
    "ridge_point_flop_per_byte": ridge_flop_per_byte
}
(ARTIFACT_DIR / "part_c_metrics.json").write_text(json.dumps(part_c, indent=2))
display(pd.DataFrame([part_c]))

log(f"Part C ridge point = {ridge_flop_per_byte:.2f} FLOP/byte")
log(f"Part C memory add AI = {mem_ai:.5f} FLOP/byte -> {mem_side}")
log(f"Part C matmul AI = {comp_ai:.2f} FLOP/byte -> {comp_side}")

del a, b
gc.collect()
torch.cuda.empty_cache()

## Git checkpoint

After reviewing the outputs and confirming the plot/table look correct:

```bash
git add 01_hw2_5_precision_bandwidth.ipynb hw2_5_artifacts/
git commit -m "HW2.5 precision and bandwidth benchmarks"
```